# Anomaly Detection on Book3 13_17.xlsx
Simple Isolation Forest notebook.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest


In [ ]:
file_path = "Book3 13_17.xlsx"
df = pd.read_excel(file_path)

df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values('Timestamp')

sensor_cols = [
    '2040-TT-6-B06K',
    '2040-FIT-6-B06U',
    '2040-PT-6-B06U'
]

df[sensor_cols] = df[sensor_cols].interpolate().bfill().ffill()
df.head()


In [ ]:
for col in sensor_cols:
    df[col+'_mean30'] = df[col].rolling(30).mean()
    df[col+'_std30'] = df[col].rolling(30).std()
    df[col+'_diff'] = df[col].diff()

df = df.bfill()

feature_cols = [c for c in df.columns if c != 'Timestamp']

X = df[feature_cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
model = IsolationForest(
    n_estimators=200,
    contamination=0.005,
    random_state=42
)

model.fit(X_scaled)

df['Anomaly'] = model.predict(X_scaled)
df['Anomaly'] = df['Anomaly'].map({1:'Normal', -1:'Anomaly'})
df['Score'] = model.decision_function(X_scaled)

df['Anomaly'].value_counts()


In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df['Timestamp'], df['2040-PT-6-B06U'])
anom = df[df['Anomaly']=='Anomaly']
plt.scatter(anom['Timestamp'], anom['2040-PT-6-B06U'], color='red', s=10)
plt.title('Detected Pressure Anomalies')
plt.show()


In [ ]:
df.to_excel('anomaly_results.xlsx', index=False)
print('Saved anomaly_results.xlsx')
